<div class="alert alert-block alert-info">

### Imports & Connection
</div>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

<div class="alert alert-block alert-info">

### Datasets
</div>

In [4]:
start_date = '20241001'
end_date = '20241031'
retention_start_date = '20241104'
retention_end_date = '20241231'

In [5]:
query_data = f'''

    WITH customer_base AS (
    
    SELECT 
        distinct
        -- yyyymmdd,
        city_name,
        -- service_obj_service_name,
        customer_id
        -- order_id
    FROM 
        orders.order_logs_fact
    WHERE 
        yyyymmdd >= '{start_date}' 
        AND yyyymmdd <= '{end_date}'
        AND city_name IN ('Bangalore', 'Chennai', 'Hyderabad')
        AND order_status = 'dropped'
        AND service_obj_service_name in ('Auto','Auto Pool','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    ),
    
    
    unlocked AS (

    SELECT 
        distinct
        -- kpsi.yyyymmdd,
        kpsi.userid as user_id,
        kpsi.rewards__transactionid as rewards_transaction_id
    FROM 
        hive.canonical.iceberg_domain_pricing_scratchcards_immutable kpsi
    WHERE 
        yyyymmdd >= '{start_date}' 
        AND yyyymmdd <= '{end_date}'
        AND rewards__type = 'EXTERNAL_REWARD'
        AND rewards__state = 'UNLOCKED'
    ),
    
    scratched AS (

    SELECT 
        distinct
        -- kpsi.yyyymmdd,
        date_format(date_parse(kpsi.yyyymmdd, '%Y%m%d')  + interval '7' day, '%Y%m%d') scratch_window,
        kpsi.userid as user_id,
        kpsi.rewards__transactionid as rewards_transaction_id
    FROM 
        hive.canonical.iceberg_domain_pricing_scratchcards_immutable kpsi
    WHERE 
        yyyymmdd >= '{start_date}' 
        AND yyyymmdd <= '{end_date}'
        AND rewards__type = 'EXTERNAL_REWARD'
        AND rewards__state = 'SCRATCHED'
    ),
    
    redeemed AS (
    
    SELECT 
        distinct
        event_props_user_id redeem_user_id
    FROM 
        clevertap.customer_rewards_immutable 
    WHERE 
       yyyymmdd >= '{start_date}' 
        AND yyyymmdd <= '{end_date}'
        AND event_props_action NOT IN ('ctaClicked', 'copyCodeClicked')
    
    ),
    
    merge AS (
    
    SELECT 
        customer_base.*,
        unlocked.rewards_transaction_id unlocked,
        scratched.rewards_transaction_id scratched,
        redeem_user_id
    FROM 
        customer_base
    LEFT JOIN 
        unlocked
        -- ON customer_base.yyyymmdd = unlocked.yyyymmdd
        ON customer_base.customer_id = unlocked.user_id
        -- AND customer_base.order_id = unlocked.rewards_transaction_id
    LEFT JOIN 
        scratched
        -- ON unlocked.yyyymmdd <= scratched.scratch_window
        ON unlocked.user_id = scratched.user_id
        -- AND unlocked.rewards_transaction_id = scratched.rewards_transaction_id
    LEFT JOIN 
        redeemed
        ON scratched.user_id = redeemed.redeem_user_id
    ),
    
    view AS (
    
    SELECT
        city_name,
        CASE 
        WHEN unlocked = 0 THEN 'Deals Not Issued'
        WHEN unlocked > 0 THEN 'Deals issused'
        -- WHEN unlocked > 0 AND scratched = 0 THEN 'Deals issused'
        -- WHEN unlocked > 0 AND scratched > 0 THEN 'GROUP-B'
        ELSE 'Check'
        END isIssued,
        CASE 
        WHEN unlocked = 0 THEN 'Deals Not Issued'
        WHEN unlocked > 0 AND scratched = 0 THEN 'Deals issused, not scratched'
        WHEN unlocked > 0 AND scratched > 0 THEN 'Deals issused, scratched'
        ELSE 'Check'
        END isScratched,
        CASE 
        WHEN unlocked = 0 THEN 'Deals Not Issued'
        WHEN unlocked > 0 AND scratched = 0 THEN 'Deals issused, not scratched'
        WHEN unlocked > 0 AND scratched > 0 AND redeemed = 0  THEN 'Deals issused, scratched not redeemed'
        WHEN unlocked > 0 AND scratched > 0 AND redeemed > 0  THEN 'Deals issused, redeemed'
        ELSE 'Check'
        END isredeemed,
        customer_id,
        unlocked,
        scratched,
        redeemed
    FROM
        (
        SELECT 
            city_name,
            customer_id,
            count(distinct unlocked) unlocked,
            count(distinct scratched) scratched,
            COUNT(distinct redeem_user_id) redeemed
        FROM 
            merge
        GROUP BY 1,2
        )
    ),

    ao_retention AS (

    SELECT 
        identity customer_id,
        
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241104' THEN 1 ELSE 0 END) w1_ao_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241111' THEN 1 ELSE 0 END) w2_ao_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241118' THEN 1 ELSE 0 END) w3_ao_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241125' THEN 1 ELSE 0 END) w4_ao_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241202' THEN 1 ELSE 0 END) w5_ao_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241209' THEN 1 ELSE 0 END) w6_ao_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241216' THEN 1 ELSE 0 END) w7_ao_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241223' THEN 1 ELSE 0 END) w8_ao_retention,
        
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('MONTH', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241101' THEN 1 ELSE 0 END) nov_ao_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('MONTH', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241201' THEN 1 ELSE 0 END) dec_ao_retention

    FROM 
        canonical.clevertap_customer_order_activity
    WHERE 
        yyyymmdd >= '{retention_start_date}'
        AND yyyymmdd <= '{retention_end_date}'
        AND order_activity_source = 'appOpen'
        AND current_city IN ('Bangalore', 'Chennai', 'Hyderabad')
        AND serviceable = 'true'
    GROUP BY 1
    
    ),

    fe_retention AS (

    SELECT 
        user_id,
        
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241104' THEN 1 ELSE 0 END) w1_fe_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241111' THEN 1 ELSE 0 END) w2_fe_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241118' THEN 1 ELSE 0 END) w3_fe_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241125' THEN 1 ELSE 0 END) w4_fe_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241202' THEN 1 ELSE 0 END) w5_fe_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241209' THEN 1 ELSE 0 END) w6_fe_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241216' THEN 1 ELSE 0 END) w7_fe_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241223' THEN 1 ELSE 0 END) w8_fe_retention,
        
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('MONTH', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241101' THEN 1 ELSE 0 END) nov_fe_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('MONTH', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241201' THEN 1 ELSE 0 END) dec_fe_retention

    FROM 
        pricing.fare_estimates_enriched
    WHERE 
        yyyymmdd >= '{retention_start_date}'
        AND yyyymmdd <= '{retention_end_date}'
        AND api_context = '/fare/estimate'
        AND city IN ('Bangalore', 'Chennai', 'Hyderabad')
        AND service_name = 'Auto'
    GROUP BY 1
    
    ),
    
    rr_net_retention AS (
    
    SELECT 
        customer_id,
        
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241104' THEN 1 ELSE 0 END) w1_gross_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241111' THEN 1 ELSE 0 END) w2_gross_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241118' THEN 1 ELSE 0 END) w3_gross_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241125' THEN 1 ELSE 0 END) w4_gross_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241202' THEN 1 ELSE 0 END) w5_gross_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241209' THEN 1 ELSE 0 END) w6_gross_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241216' THEN 1 ELSE 0 END) w7_gross_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241223' THEN 1 ELSE 0 END) w8_gross_retention,
        
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241104' AND order_status = 'dropped' THEN 1 ELSE 0 END) w1_net_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241111' AND order_status = 'dropped' THEN 1 ELSE 0 END) w2_net_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241118' AND order_status = 'dropped' THEN 1 ELSE 0 END) w3_net_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241125' AND order_status = 'dropped' THEN 1 ELSE 0 END) w4_net_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241202' AND order_status = 'dropped' THEN 1 ELSE 0 END) w5_net_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241209' AND order_status = 'dropped' THEN 1 ELSE 0 END) w6_net_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241216' AND order_status = 'dropped' THEN 1 ELSE 0 END) w7_net_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('WEEK', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241223' AND order_status = 'dropped' THEN 1 ELSE 0 END) w8_net_retention,
        
        
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('MONTH', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241101' THEN 1 ELSE 0 END) nov_gross_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('MONTH', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241201' THEN 1 ELSE 0 END) dec_gross_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('MONTH', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241101' AND order_status = 'dropped' THEN 1 ELSE 0 END) nov_net_retention,
        MAX(CASE WHEN DATE_FORMAT(DATE_TRUNC('MONTH', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') = '20241201' AND order_status = 'dropped' THEN 1 ELSE 0 END) dec_net_retention
    FROM 
        orders.order_logs_fact
    WHERE 
        yyyymmdd >= '{retention_start_date}'
        AND yyyymmdd <= '{retention_end_date}'
        AND city_name IN ('Bangalore', 'Chennai', 'Hyderabad')
        AND service_obj_service_name in ('Auto','Auto Pool','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    GROUP BY 1
    ),
    
    segment AS (
    
    SELECT  
        customer_id,
        taxi_ltr_segment,
        taxi_retention_segments,
        taxi_regularity_segment,
        taxi_fe_regularity_segment,
        taxi_recency_segment
    FROM 
        datasets.iallocator_customer_segments
    WHERE
        run_date =  '2024-10-31' 
        AND taxi_recency_segment != 'INACTIVE'
    )
    
    SELECT 
        view.*,
        taxi_ltr_segment,
        taxi_retention_segments,
        taxi_regularity_segment,
        taxi_fe_regularity_segment,
        taxi_recency_segment,

        w1_ao_retention,
        w2_ao_retention,
        w3_ao_retention,
        w4_ao_retention,
        w5_ao_retention,
        w6_ao_retention,
        w7_ao_retention,
        w8_ao_retention,
        
        w1_fe_retention,
        w2_fe_retention,
        w3_fe_retention,
        w4_fe_retention,
        w5_fe_retention,
        w6_fe_retention,
        w7_fe_retention,
        w8_fe_retention,

        w1_gross_retention,
        w2_gross_retention,
        w3_gross_retention,
        w4_gross_retention,
        w5_gross_retention,
        w6_gross_retention,
        w7_gross_retention,
        w8_gross_retention,
        
        w1_net_retention,
        w2_net_retention,
        w3_net_retention,
        w4_net_retention,
        w5_net_retention,
        w6_net_retention,
        w7_net_retention,
        w8_net_retention,

        nov_ao_retention,
        dec_ao_retention,
        nov_fe_retention,
        dec_fe_retention,
        nov_gross_retention,
        dec_gross_retention,
        nov_net_retention,
        dec_net_retention
        
    FROM 
        view
    LEFT JOIN 
        ao_retention
        ON view.customer_id = ao_retention.customer_id 
    LEFT JOIN 
        fe_retention
        ON view.customer_id = fe_retention.user_id 
    LEFT JOIN 
        rr_net_retention
        ON view.customer_id = rr_net_retention.customer_id
    LEFT JOIN 
        segment
        ON view.customer_id = segment.customer_id
'''

df = pd.read_sql(query_data, connection)

In [6]:
df.head(2)

,city_name,isIssued,isScratched,isredeemed,customer_id,unlocked,scratched,redeemed,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_fe_regularity_segment,taxi_recency_segment,w1_ao_retention,w2_ao_retention,w3_ao_retention,w4_ao_retention,w5_ao_retention,w6_ao_retention,w7_ao_retention,w8_ao_retention,w1_fe_retention,w2_fe_retention,w3_fe_retention,w4_fe_retention,w5_fe_retention,w6_fe_retention,w7_fe_retention,w8_fe_retention,w1_gross_retention,w2_gross_retention,w3_gross_retention,w4_gross_retention,w5_gross_retention,w6_gross_retention,w7_gross_retention,w8_gross_retention,w1_net_retention,w2_net_retention,w3_net_retention,w4_net_retention,w5_net_retention,w6_net_retention,w7_net_retention,w8_net_retention,nov_ao_retention,dec_ao_retention,nov_fe_retention,dec_fe_retention,nov_gross_retention,dec_gross_retention,nov_net_retention,dec_net_retention
0,Chennai,Deals issused,"Deals issused, not scratched","Deals issused, not scratched",66a1e54fd0f97db6b382772d,1,0,0,PHH,GOLD,BI_WEEKLY,MONTHLY,STATIONARY,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,0.0,1.0,NaN,NaN,NaN,NaN
1,Chennai,Deals issused,"Deals issused, not scratched","Deals issused, not scratched",64cf53f5157977ce5770d0b6,2,0,0,PHH,GOLD,BI_WEEKLY,BI_WEEKLY,STATIONARY,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [7]:
df.to_parquet('/Users/E2074/local-datasets/customer/ads-monetisation/deals-impact/data_retention.parquet', index=False)

<div class="alert alert-block alert-info">

### Parameters
</div>

### Analysis

In [9]:
df_analysis1 = df\
                .groupby(['city_name'])\
                .agg(
                    customers = ('customer_id', 'nunique'),
                    w1_ao_retention = ('customer_id', lambda x: x[df['w1_ao_retention'] == 1].nunique()),
                    w1_fe_retention = ('customer_id', lambda x: x[df['w1_fe_retention'] == 1].nunique()),
                    w1_gross_retention = ('customer_id', lambda x: x[df['w1_gross_retention'] == 1].nunique()),
                    w1_net_retention = ('customer_id', lambda x: x[df['w1_net_retention'] == 1].nunique()),
                    
                    w2_ao_retention = ('customer_id', lambda x: x[df['w2_ao_retention'] == 1].nunique()),
                    w2_fe_retention = ('customer_id', lambda x: x[df['w2_fe_retention'] == 1].nunique()),
                    w2_gross_retention = ('customer_id', lambda x: x[df['w2_gross_retention'] == 1].nunique()),
                    w2_net_retention = ('customer_id', lambda x: x[df['w2_net_retention'] == 1].nunique()),
                    
                    w3_ao_retention = ('customer_id', lambda x: x[df['w3_ao_retention'] == 1].nunique()),
                    w3_fe_retention = ('customer_id', lambda x: x[df['w3_fe_retention'] == 1].nunique()),
                    w3_gross_retention = ('customer_id', lambda x: x[df['w3_gross_retention'] == 1].nunique()),
                    w3_net_retention = ('customer_id', lambda x: x[df['w3_net_retention'] == 1].nunique()),
                    
                    w4_ao_retention = ('customer_id', lambda x: x[df['w4_ao_retention'] == 1].nunique()),
                    w4_fe_retention = ('customer_id', lambda x: x[df['w4_fe_retention'] == 1].nunique()),
                    w4_gross_retention = ('customer_id', lambda x: x[df['w4_gross_retention'] == 1].nunique()),
                    w4_net_retention = ('customer_id', lambda x: x[df['w4_net_retention'] == 1].nunique()),
                    
                    w5_ao_retention = ('customer_id', lambda x: x[df['w5_ao_retention'] == 1].nunique()),
                    w5_fe_retention = ('customer_id', lambda x: x[df['w5_fe_retention'] == 1].nunique()),
                    w5_gross_retention = ('customer_id', lambda x: x[df['w5_gross_retention'] == 1].nunique()),
                    w5_net_retention = ('customer_id', lambda x: x[df['w5_net_retention'] == 1].nunique()),
                    
                    w6_ao_retention = ('customer_id', lambda x: x[df['w6_ao_retention'] == 1].nunique()),
                    w6_fe_retention = ('customer_id', lambda x: x[df['w6_fe_retention'] == 1].nunique()),
                    w6_gross_retention = ('customer_id', lambda x: x[df['w6_gross_retention'] == 1].nunique()),
                    w6_net_retention = ('customer_id', lambda x: x[df['w6_net_retention'] == 1].nunique()),
                    
                    w7_ao_retention = ('customer_id', lambda x: x[df['w7_ao_retention'] == 1].nunique()),
                    w7_fe_retention = ('customer_id', lambda x: x[df['w7_fe_retention'] == 1].nunique()),
                    w7_gross_retention = ('customer_id', lambda x: x[df['w7_gross_retention'] == 1].nunique()),
                    w7_net_retention = ('customer_id', lambda x: x[df['w7_net_retention'] == 1].nunique()),
                    
                    w8_ao_retention = ('customer_id', lambda x: x[df['w8_ao_retention'] == 1].nunique()),
                    w8_fe_retention = ('customer_id', lambda x: x[df['w8_fe_retention'] == 1].nunique()),
                    w8_gross_retention = ('customer_id', lambda x: x[df['w8_gross_retention'] == 1].nunique()),
                    w8_net_retention = ('customer_id', lambda x: x[df['w8_net_retention'] == 1].nunique())

                    )\
                .reset_index()

df_analysis1['w1_ao %'] = round(df_analysis1['w1_ao_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w1_fe %'] = round(df_analysis1['w1_fe_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w1_rr %'] = round(df_analysis1['w1_gross_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w1_net %'] = round(df_analysis1['w1_net_retention'] * 100.00 / df_analysis1['customers'])

df_analysis1['w2_ao %'] = round(df_analysis1['w2_ao_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w2_fe %'] = round(df_analysis1['w2_fe_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w2_rr %'] = round(df_analysis1['w2_gross_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w2_net %'] = round(df_analysis1['w2_net_retention'] * 100.00 / df_analysis1['customers'])

df_analysis1['w3_ao %'] = round(df_analysis1['w3_ao_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w3_fe %'] = round(df_analysis1['w3_fe_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w3_rr %'] = round(df_analysis1['w3_gross_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w3_net %'] = round(df_analysis1['w3_net_retention'] * 100.00 / df_analysis1['customers'])

df_analysis1['w4_ao %'] = round(df_analysis1['w4_ao_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w4_fe %'] = round(df_analysis1['w4_fe_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w4_rr %'] = round(df_analysis1['w4_gross_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w4_net %'] = round(df_analysis1['w4_net_retention'] * 100.00 / df_analysis1['customers'])

df_analysis1['w5_ao %'] = round(df_analysis1['w5_ao_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w5_fe %'] = round(df_analysis1['w5_fe_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w5_rr %'] = round(df_analysis1['w5_gross_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w5_net %'] = round(df_analysis1['w5_net_retention'] * 100.00 / df_analysis1['customers'])

df_analysis1['w6_ao %'] = round(df_analysis1['w6_ao_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w6_fe %'] = round(df_analysis1['w6_fe_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w6_rr %'] = round(df_analysis1['w6_gross_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w6_net %'] = round(df_analysis1['w6_net_retention'] * 100.00 / df_analysis1['customers'])

df_analysis1['w7_ao %'] = round(df_analysis1['w7_ao_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w7_fe %'] = round(df_analysis1['w7_fe_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w7_rr %'] = round(df_analysis1['w7_gross_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w7_net %'] = round(df_analysis1['w7_net_retention'] * 100.00 / df_analysis1['customers'])

df_analysis1['w8_ao %'] = round(df_analysis1['w8_ao_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w8_fe %'] = round(df_analysis1['w8_fe_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w8_rr %'] = round(df_analysis1['w8_gross_retention'] * 100.00 / df_analysis1['customers'])
df_analysis1['w8_net %'] = round(df_analysis1['w8_net_retention'] * 100.00 / df_analysis1['customers'])


df_analysis1

,city_name,customers,w1_ao_retention,w1_fe_retention,w1_gross_retention,w1_net_retention,w2_ao_retention,w2_fe_retention,w2_gross_retention,w2_net_retention,w3_ao_retention,w3_fe_retention,w3_gross_retention,w3_net_retention,w4_ao_retention,w4_fe_retention,w4_gross_retention,w4_net_retention,w5_ao_retention,w5_fe_retention,w5_gross_retention,w5_net_retention,w6_ao_retention,w6_fe_retention,w6_gross_retention,w6_net_retention,w7_ao_retention,w7_fe_retention,w7_gross_retention,w7_net_retention,w8_ao_retention,w8_fe_retention,w8_gross_retention,w8_net_retention,w1_ao %,w1_fe %,w1_rr %,w1_net %,w2_ao %,w2_fe %,w2_rr %,w2_net %,w3_ao %,w3_fe %,w3_rr %,w3_net %,w4_ao %,w4_fe %,w4_rr %,w4_net %,w5_ao %,w5_fe %,w5_rr %,w5_net %,w6_ao %,w6_fe %,w6_rr %,w6_net %,w7_ao %,w7_fe %,w7_rr %,w7_net %,w8_ao %,w8_fe %,w8_rr %,w8_net %
0,Bangalore,2637619,1116363,1183023,1002582,884456,1155454,1213806,1035054,893486,1109735,1158979,984120,866080,1108266,1147843,968750,850707,1175627,1239746,1062198,893406,1129261,1173431,1004158,868583,1046661,1095825,925654,825400,946475,981376,809242,733352,42.0,45.0,38.0,34.0,44.0,46.0,39.0,34.0,42.0,44.0,37.0,33.0,42.0,44.0,37.0,32.0,45.0,47.0,40.0,34.0,43.0,44.0,38.0,33.0,40.0,42.0,35.0,31.0,36.0,37.0,31.0,28.0
1,Chennai,1787845,795701,842868,704933,635927,774235,806179,666678,607977,708945,731674,600384,560664,672158,680891,542388,494155,733271,762635,639599,584693,727628,748178,613718,557517,698367,731604,597363,546795,677346,701669,569845,525587,45.0,47.0,39.0,36.0,43.0,45.0,37.0,34.0,40.0,41.0,34.0,31.0,38.0,38.0,30.0,28.0,41.0,43.0,36.0,33.0,41.0,42.0,34.0,31.0,39.0,41.0,33.0,31.0,38.0,39.0,32.0,29.0
2,Hyderabad,3478486,1505394,1621875,1289064,1176319,1508130,1610991,1277708,1163044,1445738,1542066,1223243,1118428,1414430,1488124,1182501,1087306,1464998,1572464,1265585,1144943,1484031,1581476,1269139,1144339,1407625,1512533,1201949,1097319,1371566,1465155,1148309,1048712,43.0,47.0,37.0,34.0,43.0,46.0,37.0,33.0,42.0,44.0,35.0,32.0,41.0,43.0,34.0,31.0,42.0,45.0,36.0,33.0,43.0,45.0,36.0,33.0,40.0,43.0,35.0,32.0,39.0,42.0,33.0,30.0


In [ ]:
df_analysis2 = df\
                .groupby(['city_name','isIssued'])\
                .agg(
                    customers = ('customer_id', 'nunique'),
                    w1_ao_retention = ('customer_id', lambda x: x[df['w1_ao_retention'] == 1].nunique()),
                    w1_fe_retention = ('customer_id', lambda x: x[df['w1_fe_retention'] == 1].nunique()),
                    w1_gross_retention = ('customer_id', lambda x: x[df['w1_gross_retention'] == 1].nunique()),
                    w1_net_retention = ('customer_id', lambda x: x[df['w1_net_retention'] == 1].nunique()),
                    
                    w2_ao_retention = ('customer_id', lambda x: x[df['w2_ao_retention'] == 1].nunique()),
                    w2_fe_retention = ('customer_id', lambda x: x[df['w2_fe_retention'] == 1].nunique()),
                    w2_gross_retention = ('customer_id', lambda x: x[df['w2_gross_retention'] == 1].nunique()),
                    w2_net_retention = ('customer_id', lambda x: x[df['w2_net_retention'] == 1].nunique()),
                    
                    w3_ao_retention = ('customer_id', lambda x: x[df['w3_ao_retention'] == 1].nunique()),
                    w3_fe_retention = ('customer_id', lambda x: x[df['w3_fe_retention'] == 1].nunique()),
                    w3_gross_retention = ('customer_id', lambda x: x[df['w3_gross_retention'] == 1].nunique()),
                    w3_net_retention = ('customer_id', lambda x: x[df['w3_net_retention'] == 1].nunique()),
                    
                    w4_ao_retention = ('customer_id', lambda x: x[df['w4_ao_retention'] == 1].nunique()),
                    w4_fe_retention = ('customer_id', lambda x: x[df['w4_fe_retention'] == 1].nunique()),
                    w4_gross_retention = ('customer_id', lambda x: x[df['w4_gross_retention'] == 1].nunique()),
                    w4_net_retention = ('customer_id', lambda x: x[df['w4_net_retention'] == 1].nunique()),
                    
                    w5_ao_retention = ('customer_id', lambda x: x[df['w5_ao_retention'] == 1].nunique()),
                    w5_fe_retention = ('customer_id', lambda x: x[df['w5_fe_retention'] == 1].nunique()),
                    w5_gross_retention = ('customer_id', lambda x: x[df['w5_gross_retention'] == 1].nunique()),
                    w5_net_retention = ('customer_id', lambda x: x[df['w5_net_retention'] == 1].nunique()),
                    
                    w6_ao_retention = ('customer_id', lambda x: x[df['w6_ao_retention'] == 1].nunique()),
                    w6_fe_retention = ('customer_id', lambda x: x[df['w6_fe_retention'] == 1].nunique()),
                    w6_gross_retention = ('customer_id', lambda x: x[df['w6_gross_retention'] == 1].nunique()),
                    w6_net_retention = ('customer_id', lambda x: x[df['w6_net_retention'] == 1].nunique()),
                    
                    w7_ao_retention = ('customer_id', lambda x: x[df['w7_ao_retention'] == 1].nunique()),
                    w7_fe_retention = ('customer_id', lambda x: x[df['w7_fe_retention'] == 1].nunique()),
                    w7_gross_retention = ('customer_id', lambda x: x[df['w7_gross_retention'] == 1].nunique()),
                    w7_net_retention = ('customer_id', lambda x: x[df['w7_net_retention'] == 1].nunique()),
                    
                    w8_ao_retention = ('customer_id', lambda x: x[df['w8_ao_retention'] == 1].nunique()),
                    w8_fe_retention = ('customer_id', lambda x: x[df['w8_fe_retention'] == 1].nunique()),
                    w8_gross_retention = ('customer_id', lambda x: x[df['w8_gross_retention'] == 1].nunique()),
                    w8_net_retention = ('customer_id', lambda x: x[df['w8_net_retention'] == 1].nunique())
                    )\
                .reset_index()

df_analysis2['customers distr %'] = round(df_analysis2['customers']*100.00/df_analysis2.groupby(['city_name'])['customers'].transform('sum'), 2)
df_analysis2['w1_ao %'] = round(df_analysis2['w1_ao_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w1_fe %'] = round(df_analysis2['w1_fe_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w1_rr %'] = round(df_analysis2['w1_gross_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w1_net %'] = round(df_analysis2['w1_net_retention'] * 100.00 / df_analysis2['customers'])

df_analysis2['w2_ao %'] = round(df_analysis2['w2_ao_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w2_fe %'] = round(df_analysis2['w2_fe_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w2_rr %'] = round(df_analysis2['w2_gross_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w2_net %'] = round(df_analysis2['w2_net_retention'] * 100.00 / df_analysis2['customers'])

df_analysis2['w3_ao %'] = round(df_analysis2['w3_ao_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w3_fe %'] = round(df_analysis2['w3_fe_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w3_rr %'] = round(df_analysis2['w3_gross_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w3_net %'] = round(df_analysis2['w3_net_retention'] * 100.00 / df_analysis2['customers'])

df_analysis2['w4_ao %'] = round(df_analysis2['w4_ao_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w4_fe %'] = round(df_analysis2['w4_fe_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w4_rr %'] = round(df_analysis2['w4_gross_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w4_net %'] = round(df_analysis2['w4_net_retention'] * 100.00 / df_analysis2['customers'])

df_analysis2['w5_ao %'] = round(df_analysis2['w5_ao_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w5_fe %'] = round(df_analysis2['w5_fe_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w5_rr %'] = round(df_analysis2['w5_gross_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w5_net %'] = round(df_analysis2['w5_net_retention'] * 100.00 / df_analysis2['customers'])

df_analysis2['w6_ao %'] = round(df_analysis2['w6_ao_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w6_fe %'] = round(df_analysis2['w6_fe_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w6_rr %'] = round(df_analysis2['w6_gross_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w6_net %'] = round(df_analysis2['w6_net_retention'] * 100.00 / df_analysis2['customers'])

df_analysis2['w7_ao %'] = round(df_analysis2['w7_ao_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w7_fe %'] = round(df_analysis2['w7_fe_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w7_rr %'] = round(df_analysis2['w7_gross_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w7_net %'] = round(df_analysis2['w7_net_retention'] * 100.00 / df_analysis2['customers'])

df_analysis2['w8_ao %'] = round(df_analysis2['w8_ao_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w8_fe %'] = round(df_analysis2['w8_fe_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w8_rr %'] = round(df_analysis2['w8_gross_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2['w8_net %'] = round(df_analysis2['w8_net_retention'] * 100.00 / df_analysis2['customers'])
df_analysis2

In [22]:
df_analysis2v1 = df\
                .groupby(['city_name', 'taxi_retention_segments', 'isIssued'])\
                .agg(
                    customers = ('customer_id', 'nunique'),
                    w1_ao_retention = ('customer_id', lambda x: x[df['w1_ao_retention'] == 1].nunique()),
                    w1_fe_retention = ('customer_id', lambda x: x[df['w1_fe_retention'] == 1].nunique()),
                    w1_gross_retention = ('customer_id', lambda x: x[df['w1_gross_retention'] == 1].nunique()),
                    w1_net_retention = ('customer_id', lambda x: x[df['w1_net_retention'] == 1].nunique()),
                    
                    w2_ao_retention = ('customer_id', lambda x: x[df['w2_ao_retention'] == 1].nunique()),
                    w2_fe_retention = ('customer_id', lambda x: x[df['w2_fe_retention'] == 1].nunique()),
                    w2_gross_retention = ('customer_id', lambda x: x[df['w2_gross_retention'] == 1].nunique()),
                    w2_net_retention = ('customer_id', lambda x: x[df['w2_net_retention'] == 1].nunique()),
                    
                    w3_ao_retention = ('customer_id', lambda x: x[df['w3_ao_retention'] == 1].nunique()),
                    w3_fe_retention = ('customer_id', lambda x: x[df['w3_fe_retention'] == 1].nunique()),
                    w3_gross_retention = ('customer_id', lambda x: x[df['w3_gross_retention'] == 1].nunique()),
                    w3_net_retention = ('customer_id', lambda x: x[df['w3_net_retention'] == 1].nunique()),
                    
                    w4_ao_retention = ('customer_id', lambda x: x[df['w4_ao_retention'] == 1].nunique()),
                    w4_fe_retention = ('customer_id', lambda x: x[df['w4_fe_retention'] == 1].nunique()),
                    w4_gross_retention = ('customer_id', lambda x: x[df['w4_gross_retention'] == 1].nunique()),
                    w4_net_retention = ('customer_id', lambda x: x[df['w4_net_retention'] == 1].nunique()),
                    
                    w5_ao_retention = ('customer_id', lambda x: x[df['w5_ao_retention'] == 1].nunique()),
                    w5_fe_retention = ('customer_id', lambda x: x[df['w5_fe_retention'] == 1].nunique()),
                    w5_gross_retention = ('customer_id', lambda x: x[df['w5_gross_retention'] == 1].nunique()),
                    w5_net_retention = ('customer_id', lambda x: x[df['w5_net_retention'] == 1].nunique()),
                    
                    w6_ao_retention = ('customer_id', lambda x: x[df['w6_ao_retention'] == 1].nunique()),
                    w6_fe_retention = ('customer_id', lambda x: x[df['w6_fe_retention'] == 1].nunique()),
                    w6_gross_retention = ('customer_id', lambda x: x[df['w6_gross_retention'] == 1].nunique()),
                    w6_net_retention = ('customer_id', lambda x: x[df['w6_net_retention'] == 1].nunique()),
                    
                    w7_ao_retention = ('customer_id', lambda x: x[df['w7_ao_retention'] == 1].nunique()),
                    w7_fe_retention = ('customer_id', lambda x: x[df['w7_fe_retention'] == 1].nunique()),
                    w7_gross_retention = ('customer_id', lambda x: x[df['w7_gross_retention'] == 1].nunique()),
                    w7_net_retention = ('customer_id', lambda x: x[df['w7_net_retention'] == 1].nunique()),
                    
                    w8_ao_retention = ('customer_id', lambda x: x[df['w8_ao_retention'] == 1].nunique()),
                    w8_fe_retention = ('customer_id', lambda x: x[df['w8_fe_retention'] == 1].nunique()),
                    w8_gross_retention = ('customer_id', lambda x: x[df['w8_gross_retention'] == 1].nunique()),
                    w8_net_retention = ('customer_id', lambda x: x[df['w8_net_retention'] == 1].nunique())
                    )\
                .reset_index()

df_analysis2v1['customers distr %'] = round(df_analysis2v1['customers']*100.00/df_analysis2v1.groupby(['city_name', 'isIssued'])['customers'].transform('sum'), 2)
df_analysis2v1['nov retention %'] = round(df_analysis2v1['nov_retention']*100.00/df_analysis2v1['customers'])
df_analysis2v1['dec retention %'] = round(df_analysis2v1['dec_retention']*100.00/df_analysis2v1['customers'])
df_analysis2v1


KeyboardInterrupt



In [ ]:
df_analysis2v2 = df\
                .groupby(['city_name', 'taxi_regularity_segment', 'isIssued'])\
                .agg(
                    customers = ('customer_id', 'nunique'),
                    w1_ao_retention = ('customer_id', lambda x: x[df['w1_ao_retention'] == 1].nunique()),
                    w1_fe_retention = ('customer_id', lambda x: x[df['w1_fe_retention'] == 1].nunique()),
                    w1_gross_retention = ('customer_id', lambda x: x[df['w1_gross_retention'] == 1].nunique()),
                    w1_net_retention = ('customer_id', lambda x: x[df['w1_net_retention'] == 1].nunique()),
                    
                    w2_ao_retention = ('customer_id', lambda x: x[df['w2_ao_retention'] == 1].nunique()),
                    w2_fe_retention = ('customer_id', lambda x: x[df['w2_fe_retention'] == 1].nunique()),
                    w2_gross_retention = ('customer_id', lambda x: x[df['w2_gross_retention'] == 1].nunique()),
                    w2_net_retention = ('customer_id', lambda x: x[df['w2_net_retention'] == 1].nunique()),
                    
                    w3_ao_retention = ('customer_id', lambda x: x[df['w3_ao_retention'] == 1].nunique()),
                    w3_fe_retention = ('customer_id', lambda x: x[df['w3_fe_retention'] == 1].nunique()),
                    w3_gross_retention = ('customer_id', lambda x: x[df['w3_gross_retention'] == 1].nunique()),
                    w3_net_retention = ('customer_id', lambda x: x[df['w3_net_retention'] == 1].nunique()),
                    
                    w4_ao_retention = ('customer_id', lambda x: x[df['w4_ao_retention'] == 1].nunique()),
                    w4_fe_retention = ('customer_id', lambda x: x[df['w4_fe_retention'] == 1].nunique()),
                    w4_gross_retention = ('customer_id', lambda x: x[df['w4_gross_retention'] == 1].nunique()),
                    w4_net_retention = ('customer_id', lambda x: x[df['w4_net_retention'] == 1].nunique()),
                    
                    w5_ao_retention = ('customer_id', lambda x: x[df['w5_ao_retention'] == 1].nunique()),
                    w5_fe_retention = ('customer_id', lambda x: x[df['w5_fe_retention'] == 1].nunique()),
                    w5_gross_retention = ('customer_id', lambda x: x[df['w5_gross_retention'] == 1].nunique()),
                    w5_net_retention = ('customer_id', lambda x: x[df['w5_net_retention'] == 1].nunique()),
                    
                    w6_ao_retention = ('customer_id', lambda x: x[df['w6_ao_retention'] == 1].nunique()),
                    w6_fe_retention = ('customer_id', lambda x: x[df['w6_fe_retention'] == 1].nunique()),
                    w6_gross_retention = ('customer_id', lambda x: x[df['w6_gross_retention'] == 1].nunique()),
                    w6_net_retention = ('customer_id', lambda x: x[df['w6_net_retention'] == 1].nunique()),
                    
                    w7_ao_retention = ('customer_id', lambda x: x[df['w7_ao_retention'] == 1].nunique()),
                    w7_fe_retention = ('customer_id', lambda x: x[df['w7_fe_retention'] == 1].nunique()),
                    w7_gross_retention = ('customer_id', lambda x: x[df['w7_gross_retention'] == 1].nunique()),
                    w7_net_retention = ('customer_id', lambda x: x[df['w7_net_retention'] == 1].nunique()),
                    
                    w8_ao_retention = ('customer_id', lambda x: x[df['w8_ao_retention'] == 1].nunique()),
                    w8_fe_retention = ('customer_id', lambda x: x[df['w8_fe_retention'] == 1].nunique()),
                    w8_gross_retention = ('customer_id', lambda x: x[df['w8_gross_retention'] == 1].nunique()),
                    w8_net_retention = ('customer_id', lambda x: x[df['w8_net_retention'] == 1].nunique())
                    )\
                .reset_index()

df_analysis2v2['customers distr %'] = round(df_analysis2v2['customers']*100.00/df_analysis2v2.groupby(['city_name', 'isIssued'])['customers'].transform('sum'), 2)
df_analysis2v2['nov retention %'] = round(df_analysis2v2['nov_retention']*100.00/df_analysis2v2['customers'])
df_analysis2v2['dec retention %'] = round(df_analysis2v2['dec_retention']*100.00/df_analysis2v2['customers'])
df_analysis2v2

In [ ]:
df_analysis3 = df\
                .groupby(['city_name','isScratched'])\
                .agg(
                    customers = ('customer_id', 'nunique'),
                    w1_ao_retention = ('customer_id', lambda x: x[df['w1_ao_retention'] == 1].nunique()),
                    w1_fe_retention = ('customer_id', lambda x: x[df['w1_fe_retention'] == 1].nunique()),
                    w1_gross_retention = ('customer_id', lambda x: x[df['w1_gross_retention'] == 1].nunique()),
                    w1_net_retention = ('customer_id', lambda x: x[df['w1_net_retention'] == 1].nunique()),
                    
                    w2_ao_retention = ('customer_id', lambda x: x[df['w2_ao_retention'] == 1].nunique()),
                    w2_fe_retention = ('customer_id', lambda x: x[df['w2_fe_retention'] == 1].nunique()),
                    w2_gross_retention = ('customer_id', lambda x: x[df['w2_gross_retention'] == 1].nunique()),
                    w2_net_retention = ('customer_id', lambda x: x[df['w2_net_retention'] == 1].nunique()),
                    
                    w3_ao_retention = ('customer_id', lambda x: x[df['w3_ao_retention'] == 1].nunique()),
                    w3_fe_retention = ('customer_id', lambda x: x[df['w3_fe_retention'] == 1].nunique()),
                    w3_gross_retention = ('customer_id', lambda x: x[df['w3_gross_retention'] == 1].nunique()),
                    w3_net_retention = ('customer_id', lambda x: x[df['w3_net_retention'] == 1].nunique()),
                    
                    w4_ao_retention = ('customer_id', lambda x: x[df['w4_ao_retention'] == 1].nunique()),
                    w4_fe_retention = ('customer_id', lambda x: x[df['w4_fe_retention'] == 1].nunique()),
                    w4_gross_retention = ('customer_id', lambda x: x[df['w4_gross_retention'] == 1].nunique()),
                    w4_net_retention = ('customer_id', lambda x: x[df['w4_net_retention'] == 1].nunique()),
                    
                    w5_ao_retention = ('customer_id', lambda x: x[df['w5_ao_retention'] == 1].nunique()),
                    w5_fe_retention = ('customer_id', lambda x: x[df['w5_fe_retention'] == 1].nunique()),
                    w5_gross_retention = ('customer_id', lambda x: x[df['w5_gross_retention'] == 1].nunique()),
                    w5_net_retention = ('customer_id', lambda x: x[df['w5_net_retention'] == 1].nunique()),
                    
                    w6_ao_retention = ('customer_id', lambda x: x[df['w6_ao_retention'] == 1].nunique()),
                    w6_fe_retention = ('customer_id', lambda x: x[df['w6_fe_retention'] == 1].nunique()),
                    w6_gross_retention = ('customer_id', lambda x: x[df['w6_gross_retention'] == 1].nunique()),
                    w6_net_retention = ('customer_id', lambda x: x[df['w6_net_retention'] == 1].nunique()),
                    
                    w7_ao_retention = ('customer_id', lambda x: x[df['w7_ao_retention'] == 1].nunique()),
                    w7_fe_retention = ('customer_id', lambda x: x[df['w7_fe_retention'] == 1].nunique()),
                    w7_gross_retention = ('customer_id', lambda x: x[df['w7_gross_retention'] == 1].nunique()),
                    w7_net_retention = ('customer_id', lambda x: x[df['w7_net_retention'] == 1].nunique()),
                    
                    w8_ao_retention = ('customer_id', lambda x: x[df['w8_ao_retention'] == 1].nunique()),
                    w8_fe_retention = ('customer_id', lambda x: x[df['w8_fe_retention'] == 1].nunique()),
                    w8_gross_retention = ('customer_id', lambda x: x[df['w8_gross_retention'] == 1].nunique()),
                    w8_net_retention = ('customer_id', lambda x: x[df['w8_net_retention'] == 1].nunique())
                    )\
                .reset_index()

df_analysis3['customers distr %'] = round(df_analysis3['customers']*100.00/df_analysis3.groupby(['city_name'])['customers'].transform('sum'), 2)
df_analysis3['nov retention %'] = round(df_analysis3['nov_retention']*100.00/df_analysis3['customers'])
df_analysis3['dec retention %'] = round(df_analysis3['dec_retention']*100.00/df_analysis3['customers'])
df_analysis3

In [ ]:
df_analysis3v1 = df\
                .groupby(['city_name', 'taxi_retention_segments', 'isScratched'])\
                .agg(
                    customers = ('customer_id', 'nunique'),
                    w1_ao_retention = ('customer_id', lambda x: x[df['w1_ao_retention'] == 1].nunique()),
                    w1_fe_retention = ('customer_id', lambda x: x[df['w1_fe_retention'] == 1].nunique()),
                    w1_gross_retention = ('customer_id', lambda x: x[df['w1_gross_retention'] == 1].nunique()),
                    w1_net_retention = ('customer_id', lambda x: x[df['w1_net_retention'] == 1].nunique()),
                    
                    w2_ao_retention = ('customer_id', lambda x: x[df['w2_ao_retention'] == 1].nunique()),
                    w2_fe_retention = ('customer_id', lambda x: x[df['w2_fe_retention'] == 1].nunique()),
                    w2_gross_retention = ('customer_id', lambda x: x[df['w2_gross_retention'] == 1].nunique()),
                    w2_net_retention = ('customer_id', lambda x: x[df['w2_net_retention'] == 1].nunique()),
                    
                    w3_ao_retention = ('customer_id', lambda x: x[df['w3_ao_retention'] == 1].nunique()),
                    w3_fe_retention = ('customer_id', lambda x: x[df['w3_fe_retention'] == 1].nunique()),
                    w3_gross_retention = ('customer_id', lambda x: x[df['w3_gross_retention'] == 1].nunique()),
                    w3_net_retention = ('customer_id', lambda x: x[df['w3_net_retention'] == 1].nunique()),
                    
                    w4_ao_retention = ('customer_id', lambda x: x[df['w4_ao_retention'] == 1].nunique()),
                    w4_fe_retention = ('customer_id', lambda x: x[df['w4_fe_retention'] == 1].nunique()),
                    w4_gross_retention = ('customer_id', lambda x: x[df['w4_gross_retention'] == 1].nunique()),
                    w4_net_retention = ('customer_id', lambda x: x[df['w4_net_retention'] == 1].nunique()),
                    
                    w5_ao_retention = ('customer_id', lambda x: x[df['w5_ao_retention'] == 1].nunique()),
                    w5_fe_retention = ('customer_id', lambda x: x[df['w5_fe_retention'] == 1].nunique()),
                    w5_gross_retention = ('customer_id', lambda x: x[df['w5_gross_retention'] == 1].nunique()),
                    w5_net_retention = ('customer_id', lambda x: x[df['w5_net_retention'] == 1].nunique()),
                    
                    w6_ao_retention = ('customer_id', lambda x: x[df['w6_ao_retention'] == 1].nunique()),
                    w6_fe_retention = ('customer_id', lambda x: x[df['w6_fe_retention'] == 1].nunique()),
                    w6_gross_retention = ('customer_id', lambda x: x[df['w6_gross_retention'] == 1].nunique()),
                    w6_net_retention = ('customer_id', lambda x: x[df['w6_net_retention'] == 1].nunique()),
                    
                    w7_ao_retention = ('customer_id', lambda x: x[df['w7_ao_retention'] == 1].nunique()),
                    w7_fe_retention = ('customer_id', lambda x: x[df['w7_fe_retention'] == 1].nunique()),
                    w7_gross_retention = ('customer_id', lambda x: x[df['w7_gross_retention'] == 1].nunique()),
                    w7_net_retention = ('customer_id', lambda x: x[df['w7_net_retention'] == 1].nunique()),
                    
                    w8_ao_retention = ('customer_id', lambda x: x[df['w8_ao_retention'] == 1].nunique()),
                    w8_fe_retention = ('customer_id', lambda x: x[df['w8_fe_retention'] == 1].nunique()),
                    w8_gross_retention = ('customer_id', lambda x: x[df['w8_gross_retention'] == 1].nunique()),
                    w8_net_retention = ('customer_id', lambda x: x[df['w8_net_retention'] == 1].nunique())
                    )\
                .reset_index()

df_analysis3v1['customers distr %'] = round(df_analysis3v1['customers']*100.00/df_analysis3v1.groupby(['city_name', 'isScratched'])['customers'].transform('sum'), 2)
df_analysis3v1['nov retention %'] = round(df_analysis3v1['nov_retention']*100.00/df_analysis3v1['customers'])
df_analysis3v1['dec retention %'] = round(df_analysis3v1['dec_retention']*100.00/df_analysis3v1['customers'])
df_analysis3v1

In [ ]:
def resample(df):

    # Group by city_name and isScratched
    grouped = df.groupby(['city_name', 'isScratched'])
    
    # Function to resample within each group while maintaining stratification on segments
    def stratified_resample(group):
        # Stratify based on taxi_retention_segments and taxi_regularity_segment
        return group.groupby(['taxi_retention_segments', 'taxi_regularity_segment'], group_keys=False).apply(lambda x: x.sample(frac=1, random_state=42))
    
    # Apply the stratified resampling function to each group
    resampled_df = grouped.apply(stratified_resample)
    
    # Reset index to clean up the multi-level index from groupby
    resampled_df = resampled_df.reset_index(drop=True)

    return resampled_df

In [ ]:
resampled_df = resample(df)

In [ ]:
resampled_analysisv1 = resampled_df\
                .groupby(['city_name', 'taxi_retention_segments', 'isScratched'])\
                .agg(
                    customers = ('customer_id', 'nunique'),
                    w1_ao_retention = ('customer_id', lambda x: x[df['w1_ao_retention'] == 1].nunique()),
                    w1_fe_retention = ('customer_id', lambda x: x[df['w1_fe_retention'] == 1].nunique()),
                    w1_gross_retention = ('customer_id', lambda x: x[df['w1_gross_retention'] == 1].nunique()),
                    w1_net_retention = ('customer_id', lambda x: x[df['w1_net_retention'] == 1].nunique()),
                    
                    w2_ao_retention = ('customer_id', lambda x: x[df['w2_ao_retention'] == 1].nunique()),
                    w2_fe_retention = ('customer_id', lambda x: x[df['w2_fe_retention'] == 1].nunique()),
                    w2_gross_retention = ('customer_id', lambda x: x[df['w2_gross_retention'] == 1].nunique()),
                    w2_net_retention = ('customer_id', lambda x: x[df['w2_net_retention'] == 1].nunique()),
                    
                    w3_ao_retention = ('customer_id', lambda x: x[df['w3_ao_retention'] == 1].nunique()),
                    w3_fe_retention = ('customer_id', lambda x: x[df['w3_fe_retention'] == 1].nunique()),
                    w3_gross_retention = ('customer_id', lambda x: x[df['w3_gross_retention'] == 1].nunique()),
                    w3_net_retention = ('customer_id', lambda x: x[df['w3_net_retention'] == 1].nunique()),
                    
                    w4_ao_retention = ('customer_id', lambda x: x[df['w4_ao_retention'] == 1].nunique()),
                    w4_fe_retention = ('customer_id', lambda x: x[df['w4_fe_retention'] == 1].nunique()),
                    w4_gross_retention = ('customer_id', lambda x: x[df['w4_gross_retention'] == 1].nunique()),
                    w4_net_retention = ('customer_id', lambda x: x[df['w4_net_retention'] == 1].nunique()),
                    
                    w5_ao_retention = ('customer_id', lambda x: x[df['w5_ao_retention'] == 1].nunique()),
                    w5_fe_retention = ('customer_id', lambda x: x[df['w5_fe_retention'] == 1].nunique()),
                    w5_gross_retention = ('customer_id', lambda x: x[df['w5_gross_retention'] == 1].nunique()),
                    w5_net_retention = ('customer_id', lambda x: x[df['w5_net_retention'] == 1].nunique()),
                    
                    w6_ao_retention = ('customer_id', lambda x: x[df['w6_ao_retention'] == 1].nunique()),
                    w6_fe_retention = ('customer_id', lambda x: x[df['w6_fe_retention'] == 1].nunique()),
                    w6_gross_retention = ('customer_id', lambda x: x[df['w6_gross_retention'] == 1].nunique()),
                    w6_net_retention = ('customer_id', lambda x: x[df['w6_net_retention'] == 1].nunique()),
                    
                    w7_ao_retention = ('customer_id', lambda x: x[df['w7_ao_retention'] == 1].nunique()),
                    w7_fe_retention = ('customer_id', lambda x: x[df['w7_fe_retention'] == 1].nunique()),
                    w7_gross_retention = ('customer_id', lambda x: x[df['w7_gross_retention'] == 1].nunique()),
                    w7_net_retention = ('customer_id', lambda x: x[df['w7_net_retention'] == 1].nunique()),
                    
                    w8_ao_retention = ('customer_id', lambda x: x[df['w8_ao_retention'] == 1].nunique()),
                    w8_fe_retention = ('customer_id', lambda x: x[df['w8_fe_retention'] == 1].nunique()),
                    w8_gross_retention = ('customer_id', lambda x: x[df['w8_gross_retention'] == 1].nunique()),
                    w8_net_retention = ('customer_id', lambda x: x[df['w8_net_retention'] == 1].nunique())
                    )\
                .reset_index()

resampled_analysisv1['customers distr %'] = round(resampled_analysisv1['customers']*100.00/resampled_analysisv1.groupby(['city_name', 'isScratched'])['customers'].transform('sum'), 2)
resampled_analysisv1['nov retention %'] = round(resampled_analysisv1['nov_retention']*100.00/resampled_analysisv1['customers'])
resampled_analysisv1['dec retention %'] = round(resampled_analysisv1['dec_retention']*100.00/resampled_analysisv1['customers'])
resampled_analysisv1

In [ ]:
resampled_df.to_parquet('/Users/E2074/local-datasets/customer/ads-monetisation/deals-impact/resampled_data_v1.parquet', index=False)